# Collate the bone metastasis information

In [ ]:
import os 

SAVE_DIR = f"{os.getenv('nnUNet_results')}/{{dataset}}/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/{{fold_name}}/bone_analysis/"

# Helper function to get the mets and metrics files that are generated by bone_analysis/lesion_metrics_new.sh
# This can be run instead of the normal nnUNet.evaluation file 

def get_mets_files(dataset, fold_name):
    mets_dir = SAVE_DIR.format(dataset=dataset, fold_name=fold_name)
    print(mets_dir)

    if not os.path.exists(mets_dir):
        print(f"Directory {mets_dir} does not exist.")
        return []
    
    mets_files = [os.path.join(mets_dir, f) for f in os.listdir(mets_dir) if f.endswith('mets.csv')]
    metrics_files = [f.replace('mets.csv', 'mets_metrics.csv') for f in mets_files]

    print(f"Found {len(mets_files)} METS files in {mets_dir}.")
    return mets_files, metrics_files

In [ ]:
import pandas as pd
# helper functions to preprocess the loading of the files

def _preprocess_mets_file(mets_file):
    try:
        mets_df = pd.read_csv(mets_file)
        case_name = os.path.basename(mets_file).replace('_mets.csv', '')
        # for the columns with Volume, divide by 1000 to convert from mm^3 to cm^3
        mets_df['file'] = case_name
        for col in mets_df.columns:
            if 'Volume' in col or 'mm3' in col:
                mets_df[col] = mets_df[col] / 1000.0
        return mets_df

    except Exception as e:
        print(f"Error reading {mets_file}: {e}")
        return None

def _preprocess_metrics_file(metrics_file): 
    try: 
        metrics_df = pd.read_csv(metrics_file)
    
        for col in metrics_df.columns:
            if 'vol' in col or 'mm3' in col:
                metrics_df[col] = metrics_df[col] / 1000.0
        
        metrics_df['file'] = metrics_df['file'].str.replace('_0000', '')
    except Exception as e:
        print(f"Error reading {metrics_file}: {e}")
        return None

    return metrics_df


def hr_organ_analysis(mets_df):
    if mets_df is None:
    turn None, None
    lung_df = mets_df.loc[mets_df['Description'].str.contains('lung', case=False, na=False)     ]
    liver_df = mets_df.loc[mets_df['Description'] == 'liver']
    return lung_df, liver_df

def hr_bone_analysis(mets_df):
    if mets_df is None:
        return None, None
    priority_bones  = [25,26,27,31,32,43,44,69,70,71,72,73,74,75,76,77,78]
    bones_df = mets_df.loc[mets_df['Type'].str.contains('bone', case=False, na=False)]
    hr_bones_df = bones_df.loc[bones_df['Label'].isin(priority_bones)]
    return bones_df, hr_bones_df



In [3]:


import os
import numpy as np

def convert_location_to_case_name(file_locations):
    case_names = []
    for path in file_locations:
        parts = path.split('/')
        # Extract patient ID from PETCT_xxxx
        patient_folder = parts[2]  # e.g., 'PETCT_0011f3deaf'
        patient_id = patient_folder.replace('PETCT_', '')
        
        # Extract scan folder (date-NA-scan_type-number)
        scan_folder = parts[3]  # e.g., '03-23-2003-NA-PET-CT Ganzkoerper...-10445'
        
        # Create CSV filename
        case_name = f"fdg_{patient_id}_{scan_folder}"
        case_names.append(case_name)
    
    return np.array(case_names)



In [4]:
def get_global_metrics(mets_files: list, metrics_files: list): 
    """
    Returns global metrics (dice, fp vol, fn vol)
    mets metrics (individual stats per organ per case)
    liver metrics (individual stats filtered for liver)
    lung metrics (individual stats filtered for lung)
    hr bones metrics (individual stats filtered for high risk bones)
    bones metrics (individual stats filtered for all bones)

    """
    global_mets = pd.DataFrame()
    global_liver = pd.DataFrame()
    global_lung = pd.DataFrame()
    global_bones = pd.DataFrame()
    global_hr_bones = pd.DataFrame()
    global_metrics = pd.DataFrame()
    priority_bones  = [25,26,27,31,32,43,44,69,70,71,72,73,74,75,76,77,78]

    for idx, _ in enumerate(mets_files):
        temp_mets_df = _preprocess_mets_file(mets_files[idx])
        temp_metrics_df = _preprocess_metrics_file(metrics_files[idx])
        
        lung_df, liver_df = hr_organ_analysis(temp_mets_df)
        bones_df, hr_bones_df = hr_bone_analysis(temp_mets_df)
        
        global_metrics = pd.concat([global_metrics, temp_metrics_df], ignore_index=True)
        global_mets = pd.concat([global_mets, temp_mets_df], ignore_index=True)
        global_liver = pd.concat([global_liver, liver_df], ignore_index=True)
        global_lung = pd.concat([global_lung, lung_df], ignore_index=True)
        global_bones = pd.concat([global_bones, bones_df], ignore_index=True)
        global_hr_bones = pd.concat([global_hr_bones, hr_bones_df], ignore_index=True)
    
    return global_metrics, global_mets, global_liver, global_lung, global_bones, global_hr_bones

class Metrics:    
    def __init__(self, dataset, fold, global_metrics, global_mets, global_liver, global_lung, global_bones, global_hr_bones):
        self.dataset = dataset
        self.fold = fold
        
        self.global_metrics = self.add_diagnosis(global_metrics)
        self.global_mets = self.add_diagnosis(global_mets)
        
        self.global_liver = self.global_mets[self.global_mets['Description'] == 'liver']
        self.global_lung = self.global_mets[self.global_mets['Description'].str.contains('lung', case=False, na=False)]
        
        self.global_bones = global_bones
        self.global_hr_bones = global_hr_bones
                    
        self.adjust_liver(overlap_threshold=0.4)
        self.adjust_lung(overlap_threshold=0.4)
        self.adjust_bones(overlap_threshold=0.4)
        self.adjust_hr_bones(overlap_threshold=0.4)

    def add_diagnosis(self, df): 
        fdg_metadata = pd.read_csv('fdg_metadata.csv')
        fdg_metadata['file'] = convert_location_to_case_name(fdg_metadata['File Location'].values)
        df = df.merge(
            fdg_metadata[['file', 'diagnosis']].drop_duplicates(subset=['file', 'diagnosis']),
            on='file',
            how='left'
            )
        
        # all PSMA cases are prostate cancer
        df.loc[df['file'].str.contains('psma', case=False, na=False), 'diagnosis'] = 'PROSTATE_CANCER'
        return df
    

    def only_diseased(self, df=None):
        if df is None:
            df = self.global_metrics         
        return df.loc[df['file'].isin(df.groupby('file')['gt_lesion_count'].sum().loc[lambda x: x > 0].index)]

    def get_fold_level_metrics(self, df = None): 
        if df is None:
            df = self.global_metrics
        return{
            "Dice": df['dice_sc'].mean(),
            "FP_vol_avg (cm3)": df['false_pos_vol'].mean(),
            "FN_vol_avg (cm3)": df['false_neg_vol'].mean(),
            "FP_vol_total (cm3)": df['false_pos_vol'].sum(),
            "FN_vol_total (cm3)": df['false_neg_vol'].sum(), 
            "Total predicted lesions": df['pred_lesion_count'].sum()
        }
    
    def get_mets_level_metrics(self, mets_df):
        return {
            "Dice": mets_df['Dice_Score'].mean(),
            "FP_vol_avg (cm3)": mets_df['FP_vol_mm3'].mean(),
            "FN_vol_avg (cm3)": mets_df['FN_vol_mm3'].mean(),
            "FP_vol_total (cm3)": mets_df['FP_vol_mm3'].sum(),
            "FN_vol_total (cm3)": mets_df['FN_vol_mm3'].sum()
            }
    
    def adjust_to_diseased(self, df): 
        return df.loc['file']

    def adjust_liver(self, overlap_threshold=0.1):
        self.global_liver = self.global_mets[
        (self.global_mets['Description'] == 'liver') &
        ((self.global_mets['Overlap_Volume_ground'] > overlap_threshold) | (self.global_mets['Overlap_Volume_pred'] > overlap_threshold))
    ]
    
    def adjust_lung(self, overlap_threshold=0.1):
        self.global_lung = self.global_mets[
        (self.global_mets['Description'].str.contains('lung', case=False, na=False)) &
        ((self.global_mets['Overlap_Volume_ground'] > overlap_threshold) | (self.global_mets['Overlap_Volume_pred'] > overlap_threshold))
    ]
    
    def adjust_bones(self, overlap_threshold=0.1):
        self.global_bones = self.global_mets[
        (self.global_mets['Type'].str.contains('bone', case=False, na=False)) &
        ((self.global_mets['Overlap_Volume_ground'] > overlap_threshold) | (self.global_mets['Overlap_Volume_pred'] > overlap_threshold))
    ]
        
    def adjust_hr_bones(self, overlap_threshold=0.1):
        self.global_hr_bones = self.global_mets[
        (self.global_mets['Type'].str.contains('bone', case=False, na=False)) &
        (self.global_mets['Label'].isin([25,26,27,31,32,43,44,69,70,71,72,73,74,75,76,77,78])) & # CHECK THIS 
        ((self.global_mets['Overlap_Volume_ground'] > overlap_threshold) | (self.global_mets['Overlap_Volume_pred'] > overlap_threshold))
    ]
        
            
    

In [5]:
DATASET = "Dataset999_AutoPet"
FOLD_NAME = "fold_6"

display_metrics = []
display_fdg_metrics = []
display_psma_metrics = []

folds = [f"fold_{i}" for i in range(0, 9)]
for fold in folds: 
    try: 
        mets_files, metric_files = get_mets_files(DATASET, fold)
        dat = Metrics(DATASET, FOLD_NAME, *get_global_metrics(mets_files, metric_files))
        display_metrics.append(dat.get_fold_level_metrics())
        display_fdg_metrics.append(dat.get_fold_level_metrics(dat.global_metrics.loc[dat.global_metrics['file'].str.contains('fdg', case=False, na=False)]))
        display_psma_metrics.append(dat.get_fold_level_metrics(dat.global_metrics.loc[dat.global_metrics['file'].str.contains('psma', case=False, na=False)]))
    except Exception as e:
        print(f"Error occurred while processing fold {fold}: {e}")


/scratch4/workspace/f007g3j_dartmouth_edu-simple/nnUNet_data/nnUNet_results/Dataset999_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_0/bone_analysis/
Found 321 METS files in /scratch4/workspace/f007g3j_dartmouth_edu-simple/nnUNet_data/nnUNet_results/Dataset999_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_0/bone_analysis/.
/scratch4/workspace/f007g3j_dartmouth_edu-simple/nnUNet_data/nnUNet_results/Dataset999_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_1/bone_analysis/
Found 321 METS files in /scratch4/workspace/f007g3j_dartmouth_edu-simple/nnUNet_data/nnUNet_results/Dataset999_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_1/bone_analysis/.
/scratch4/workspace/f007g3j_dartmouth_edu-simple/nnUNet_data/nnUNet_results/Dataset999_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_2/bone_analysis/
Found 321 METS files in /scratch4/workspace/f

In [6]:
dat.global_liver.loc[dat.global_liver['Overlap_Percentage_ground'] > 1]

,Label,Description,Count,Type,Priority,Overlap_Count_ground,Overlap_Volume_ground,Overlap_Percentage_ground,High Risk_ground,Overlap_Count_pred,...,Intersection_mm3,dice_sc,false_pos_vol,false_neg_vol,pred_lesion_count,gt_lesion_count,false_detected_lesion_count,false_missed_lesion_count,file,diagnosis
917,5,liver,21550,Organ,True,538,44.621574,2.50,True,108,...,8.211033,0.306502,0.000000,18910.258257,12,29,0,17,psma_505ebb983b71c700_2015-05-29,PROSTATE_CANCER
2079,5,liver,157689,Organ,True,17009,211.609319,10.79,True,7289,...,89.102589,0.589514,0.000000,49.764082,34,7,0,4,fdg_92c5c944a5_07-15-2002-NA-PET-CT Ganzkoerpe...,LUNG_CANCER
2411,5,liver,30541,Organ,True,8888,737.168313,29.10,True,4443,...,295.265436,0.534093,1990.553501,5059.323481,31,14,5,6,psma_505ebb983b71c700_2015-11-07,PROSTATE_CANCER
10960,5,liver,234264,Organ,True,118223,1470.814773,50.47,True,43536,...,525.720206,0.522469,186.615308,1928.358186,48,5,2,1,fdg_15a205ffcc_08-19-2005-NA-PET-CT Ganzkoerpe...,MELANOMA
13035,5,liver,113856,Organ,True,1445,17.977275,1.27,True,293,...,3.495927,0.323360,24.882041,0.000000,7,2,1,0,fdg_99a7bfad23_08-26-2000-NA-PET-CT Ganzkoerpe...,LUNG_CANCER
14529,5,liver,107843,Organ,True,1636,20.353510,1.52,True,1695,...,18.362946,0.886220,0.000000,49.764082,6,8,0,2,fdg_dc6174cb5d_03-29-2003-NA-PET-CT Ganzkoerpe...,MELANOMA
18015,5,liver,121426,Organ,True,2566,31.923659,2.11,True,2178,...,23.625498,0.800590,0.000000,0.000000,2,2,0,0,fdg_47cd731006_01-25-2002-NA-PET-CT Ganzkoerpe...,MELANOMA
19343,5,liver,134482,Organ,True,4323,53.782532,3.21,True,1746,...,20.527684,0.543747,0.000000,696.697151,7,8,0,2,fdg_d4b2ff9721_12-16-2001-NA-PET-CT Ganzkoerpe...,LUNG_CANCER
20505,5,liver,181858,Organ,True,4043,50.299046,2.22,True,2040,...,24.894482,0.657899,0.000000,2077.650433,5,5,0,1,fdg_7a77b26403_10-20-2000-NA-PET-CT Ganzkoerpe...,LUNG_CANCER
22082,5,liver,113709,Organ,True,5201,64.705748,4.57,True,3748,...,43.755069,0.786010,0.000000,0.000000,8,8,0,0,fdg_09ee00bdc6_09-30-2005-NA-PET-CT Ganzkoerpe...,MELANOMA


In [7]:
dat.global_metrics['diagnosis'].value_counts()

diagnosis
PROSTATE_CANCER    122
NEGATIVE           101
LUNG_CANCER         35
MELANOMA            31
LYMPHOMA            30
Name: count, dtype: int64

In [ ]:
pd.DataFrame(display_metrics).round(3)

,Dice,FP_vol_avg (cm3),FN_vol_avg (cm3),FP_vol_total (cm3),FN_vol_total (cm3),Total predicted lesions
0,0.378,7.349,11.22,2359.008,3601.615,5468


In [ ]:
dat.get_fold_level_metrics()

{'Dice': np.float64(0.3777544313187151),
 'FP_vol_avg (cm3)': np.float64(7.348934114532534),
 'FN_vol_avg (cm3)': np.float64(11.219983074639643),
 'FP_vol_total (cm3)': np.float64(2359.0078507649437),
 'FN_vol_total (cm3)': np.float64(3601.614566959325),
 'Total predicted lesions': np.int64(5468)}

In [ ]:
pd.DataFrame(display_psma_metrics).round(3)

,Dice,FP_vol_avg (cm3),FN_vol_avg (cm3),FP_vol_total (cm3),FN_vol_total (cm3),Total predicted lesions
0,0.483,10.152,15.344,1238.594,1871.939,3091


# Edit the existing files just in case

In [ ]:
DATASET = "Dataset111_AutoPet"
FOLD_NAME = "fold_2"

mets_files, metric_files = get_mets_files(DATASET, FOLD_NAME)


/scratch4/workspace/f007g3j_dartmouth_edu-simple/nnUNet_data/nnUNet_results/Dataset111_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_2/bone_analysis/
Found 321 METS files in /scratch4/workspace/f007g3j_dartmouth_edu-simple/nnUNet_data/nnUNet_results/Dataset111_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_2/bone_analysis/.


In [ ]:
import pandas as pd 

for file in mets_files: 
    df = pd.read_csv(file)


SyntaxError: expected ':' (3271154009.py, line 5)